In [ ]:
from fastai.vision.all import *
import PIL
from tqdm import tqdm
import seaborn as sns
import numpy as np
import glob
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import torch, gc
gc.collect()
torch.cuda.empty_cache()
import cv2
import os
import dill
import pydicom

In [ ]:
# Function to get pixel spacing ratio (pixels to centimeters) from a DICOM file
def get_pixel_to_cm_ratio(dicom_file):
    ds = pydicom.dcmread(dicom_file)  # Read the DICOM file

    # Check if 'ExposedArea' tag is present in the DICOM metadata
    if 'ExposedArea' in ds:
        exposed_area = ds.ExposedArea  # Usually in millimeters (mm)
        rows = ds.Rows                 # Number of pixel rows in image
        columns = ds.Columns           # Number of pixel columns in image

        # ExposedArea typically contains two values: [width_mm, height_mm]
        width_mm = exposed_area[0]
        height_mm = exposed_area[1]

        # Calculate pixel spacing in mm (width per pixel and height per pixel)
        pixel_spacing_mm = [width_mm / columns, height_mm / rows]

        # Convert pixel spacing from mm to cm (divide by 10)
        pixel_spacing_cm = [spacing / 10.0 for spacing in pixel_spacing_mm]
        # Return pixel spacing in cm along with image dimensions and physical width/height
        return pixel_spacing_cm, rows, columns, width_mm, height_mm
    else:
        # Raise an error if ExposedArea is missing, which means no pixel spacing info available
        raise ValueError("No suitable pixel spacing information found in DICOM file")


# Import necessary libraries (note: imports usually placed at top of file)
import os
import pydicom
from PIL import Image
import numpy as np


# Function to convert a single DICOM file to JPG image
def convert_dicom_to_jpg(dicom_path, output_path):
    try:
        # Read the DICOM file
        dicom_file = pydicom.dcmread(dicom_path)
        # Extract pixel data array from the DICOM file
        pixel_array = dicom_file.pixel_array

        # Normalize pixel values to 0-255 range for image saving
        pixel_array = ((pixel_array - np.min(pixel_array)) / 
                       (np.max(pixel_array) - np.min(pixel_array)) * 255).astype(np.uint8)

        # Create a PIL Image object from the normalized pixel array
        image = Image.fromarray(pixel_array)
        # Save the image to the specified output path as JPG
        image.save(output_path)
    except Exception as e:
        # Print an error message if conversion fails, but do not stop the process
        print(f"Failed to convert {dicom_path}: {e}")


# Function to walk through all folders/files under a root directory,
# find all DICOM files, and convert them to JPG format in the same folder
def process_directory(root_dir):
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            # Check if file extension is '.dcm' (case-insensitive)
            if file.lower().endswith('.dcm'):
                dicom_path = os.path.join(subdir, file)
                # Output JPG path with same filename but '.jpg' extension
                jpg_path = os.path.join(subdir, os.path.splitext(file)[0] + '.jpg')
                # Convert the DICOM file to JPG
                convert_dicom_to_jpg(dicom_path, jpg_path)


In [ ]:
import pandas as pd
import pydicom
import numpy as np

# Function to extract pixel spacing (in cm) and image dimensions from a DICOM file
def get_pixel_to_cm_ratio(dicom_file):
    ds = pydicom.dcmread(dicom_file)  # Read DICOM file
    
    if 'ExposedArea' in ds:
        exposed_area = ds.ExposedArea  # Typically physical dimensions in mm (width, height)
        rows = ds.Rows                 # Number of pixel rows in image
        columns = ds.Columns           # Number of pixel columns in image

        # ExposedArea usually contains two values: [width_mm, height_mm]
        width_mm = exposed_area[0]
        height_mm = exposed_area[1]

        # Calculate pixel spacing in mm (size per pixel)
        pixel_spacing_mm = [width_mm / columns, height_mm / rows]

        # Convert from mm to cm (divide by 10)
        pixel_spacing_cm = [spacing / 10.0 for spacing in pixel_spacing_mm]
        
        # Return pixel spacing (cm), image dimensions in pixels, and physical sizes (mm)
        return pixel_spacing_cm, rows, columns, width_mm, height_mm
    else:
        # Raise an error if ExposedArea is not present in DICOM metadata
        raise ValueError("No suitable pixel spacing information found in DICOM file")

# Load CSV file containing paths to DICOM files
csv_path = '/home1/09601/mz868/full_body_transparent_path.csv'
df = pd.read_csv(csv_path)

# Drop rows where the 'path_full_body_transparent_dxa' column is NaN (missing)
df = df.dropna(subset=['path_full_body_transparent_dxa'])

# Prepare an empty DataFrame to collect metadata
# NOTE: Initial row with arbitrary values is created here - probably leftover from testing
metadata = pd.DataFrame(
    data=np.array([['1.111.11', '1.111.11', 1.1, 1.1, 1.1, 1.1, 1.1, 1.1]]),
    columns=['eid', 'image_name', 'pixel_spacing_cm_X', 'pixel_spacing_cm_Y', 'rows', 'columns', 'width_mm', 'height_mm']
)

# Iterate through each row in the CSV dataframe
for index, row in df.iterrows():
    # Construct full file path to DICOM image
    dicom_file = '/scratch1/07880/devansh/knee_OA/' + row['path_full_body_transparent_dxa']
    
    try:
        # Extract pixel spacing and image metadata from the DICOM file
        pixel_spacing_cm, rows, columns, width_mm, height_mm = get_pixel_to_cm_ratio(dicom_file)
        
        # Create a dictionary for the metadata of this file
        metadata_row = {
            'eid': row['Patient EID'],
            'image_name': row['path_full_body_transparent_dxa'],
            'pixel_spacing_cm_X': pixel_spacing_cm[0],
            'pixel_spacing_cm_Y': pixel_spacing_cm[1],
            'rows': rows,
            'columns': columns,
            'width_mm': width_mm,
            'height_mm': height_mm
        }
        # Append the metadata row to the metadata DataFrame
        metadata = metadata.append(metadata_row, ignore_index=True)
        
        print(f"Processed {row['path_full_body_transparent_dxa']}")
    
    except ValueError as e:
        # Print error message but continue processing other files
        print(f"Error processing {row['path_full_body_transparent_dxa']}: {e}")

# Save the collected metadata to a new CSV file
output_csv_path = '/home1/09601/mz868/dicom_img_metadata.csv'
metadata.to_csv(output_csv_path, index=False)


In [ ]:
# Imports from fastai and other image processing libraries
from fastai.basics import *
from fastai.callback.all import *
from fastai.vision.all import *
# from fastai.medical.imaging import *  # commented out, not currently used

import pydicom
import pandas as pd
import os
import random
import numpy as np

from scipy import ndimage
from skimage import morphology
from skimage.transform import resize
import cv2

# Function to add padding to a 2D image numpy array to reach new dimensions
# Pads the image centered in a larger canvas filled with either zeros or white (255)
def add_pad(image, new_height, new_width, white):
    height, width = image.shape

    # Initialize blank canvas with zeros
    final_image = np.zeros((new_height, new_width))
    if white:
        # Fill canvas with 255 if white padding is requested
        final_image.fill(255)

    # Calculate padding offsets to center original image
    pad_left = int((new_width - width) / 2)
    pad_top = int((new_height - height) / 2)
   
    # Place the original image into the centered location on the blank canvas
    final_image[pad_top:pad_top + height, pad_left:pad_left + width] = image
   
    return final_image

# Load CSV with DICOM file paths
csv_path = '/home1/09601/mz868/dicomantijoin.csv'
df = pd.read_csv(csv_path)

# Drop rows with missing DICOM file path entries
df = df.dropna(subset=['path_full_body_transparent_dxa'])

small_files = []  # List to track files padded to smaller size
big_files = []    # List to track files padded to bigger size

# Iterate over all rows in CSV
for index, row in df.iterrows():
    # Construct full DICOM file path based on relative path in CSV
    dicom_file = '/scratch1/09601/mz868/' + row['path_full_body_transparent_dxa']
    
    # Read DICOM file
    xray_sample = pydicom.dcmread(dicom_file)
    image = xray_sample.pixel_array
    
    # Get image dimensions from DICOM tags
    rows = xray_sample[0x00280010].value  # Number of rows (height)
    cols = xray_sample[0x00280011].value  # Number of columns (width)
    
    # Condition 1: If image width is 272 pixels and height between 661 and 815 inclusive,
    # pad image to 816 x 288 with black background (white=False)
    if cols == 272 and rows > 660 and rows < 816:
        final_image = add_pad(image, 816, 288, False)
        small_files.append(dicom_file)  # Track this file as small-sized padding
        
        pixel_array_np = final_image
        f = Path(dicom_file).name  # Extract filename
        img = f.replace('.dcm', '.jpg')  # Replace extension
        
        # Directory to save padded JPGs for small size
        jpg_directory = "/scratch1/09601/mz868/unsegmentedImages/dicom820/"
        
        # Save padded image as JPG
        cv2.imwrite(os.path.join(jpg_directory, img), pixel_array_np)
        
        new_file_name = Path(jpg_directory/Path(img))  # Assigned but not used further here
    
    # Condition 2: If image width is between 301 and 381 pixels (inclusive),
    # pad image to 960 x 384 with white background (white=True)
    elif cols > 300 and cols < 382:
        final_image = add_pad(image, 960, 384, True)
        big_files.append(dicom_file)  # Track this file as big-sized padding
        
        pixel_array_np = final_image
        f = Path(dicom_file).name
        img = f.replace('.dcm', '.jpg')
        
        # Directory to save padded JPGs for big size
        jpg_directory = "/scratch1/09601/mz868/unsegmentedImages/dicom960/"
        
        # Save padded image as JPG
        cv2.imwrite(os.path.join(jpg_directory, img), pixel_array_np)
        
        new_file_name = Path(jpg_directory/Path(img))  # Assigned but not used further here

# After loop, save lists of processed files to CSV for record keeping

small_df = pd.DataFrame()
small_df["File"] = small_files
small_df.to_csv("/scratch1/09601/mz868/unsegmentedImages/dicom820/Patient_EID.csv", sep='\t', index=False)
# Note: The comment at end of line suggests an old path - possibly leftover

big_df = pd.DataFrame()
big_df["File"] = big_files
big_df.to_csv("/scratch1/09601/mz868/unsegmentedImages/dicom960/Patient_EID.csv", sep='\t', index=False)
# Note: Same as above, old path in comment



In [ ]:
# Define paths for images, labels, and class codes (some commented out alternate paths)
img_path = "/work2/09601/mz868/frontera/scoliosisTraining/images/"
lbl_path = "/work2/09601/mz868/frontera/scoliosisTraining/labels/"
class_path = "/work2/09601/mz868/frontera/scoliosisTraining/codes.txt"
# img_path = "/home1/09601/mz868/.fastai/data/spineSegment/images/"
# lbl_path = "/home1/09601/mz868/.fastai/data/spineSegment/labels/"
# class_path = "/home1/09601/mz868/.fastai/data/spineSegment/codes.txt"

# Loop over all files in the label directory
for i in range(len(os.listdir(lbl_path))):
    # Read the grayscale label image as a numpy array
    gray_value = np.array(Image.open(f'{lbl_path}{os.listdir(lbl_path)[i]}'))

    # Create a 3-channel image by duplicating the grayscale image into R,G,B
    rgb_value = cv2.merge((gray_value, gray_value, gray_value))

    # Convert the 3-channel image back to grayscale explicitly (redundant here)
    gray_img = cv2.cvtColor(rgb_value, cv2.COLOR_BGR2GRAY)

    # Save the resulting grayscale image to a new folder
    # Path hardcoded, make sure folder exists
    cv2.imwrite(f'/work2/09601/mz868/frontera/scoliosisTraining/820GrayVersion/{os.listdir(lbl_path)[i]}', gray_img)


In [ ]:
# Paths for images, labels, and class codes (some commented out alternatives)
img_path = "/work2/09601/mz868/frontera/scoliosisTraining/images/"
lbl_path = "/work2/09601/mz868/frontera/scoliosisTraining/GrayVersion/"
class_path = "/work2/09601/mz868/frontera/scoliosisTraining/codes.txt"
# img_path = "/home1/09601/mz868/.fastai/data/spineSegment/images/"
# lbl_path = "/home1/09601/mz868/.fastai/data/spineSegment/labels/"
# class_path = "/home1/09601/mz868/.fastai/data/spineSegment/codes.txt"

# Get all image file paths from the images directory
fnames = get_image_files(img_path)

# Define the path to save models or other outputs
path = "/work2/09601/mz868/frontera/modelSaves/"

# Create the DataLoaders for segmentation
dls = SegmentationDataLoaders.from_label_func(
    path,                   # path used for relative file referencing or saving
    bs=4,                   # batch size
    item_tfms=Resize((960,384)),  # Transform to resize input images and masks to 960x384 pixels
    fnames=fnames,          # list of image files
    label_func = lambda x: f'{lbl_path}{x.stem}.png',  # function to find corresponding label file for each image
    # Alternative label_func options commented out:
    # label_func = lambda x: f'{lbl_path}{x.stem}.jpg',
    # label_func = lambda x: f'{lbl_path}{x.stem}{x.suffix}', # if x.suffix == ".jpg" else f'{lbl_path}{x.stem}.png',
    codes = np.loadtxt(class_path, dtype=str)  # Load class codes from file as string array
)

# Display a batch of images and their segmentation masks
dls.show_batch()

In [ ]:
# Create a UNet learner for segmentation using a ResNet34 backbone
learn = unet_learner(
    dls,               # the DataLoaders created earlier
    resnet34,          # pretrained encoder model
    metrics=foreground_acc  # metric to track (accuracy of foreground pixels)
)

# Fine-tune the model for 12 epochs (with a frozen encoder initially, then unfreeze)
learn.fine_tune(12)

# Evaluate the model on the validation set and print results
learn.validate()

# Show example predictions alongside inputs and targets, max 4 images,
# with a large figure size for visibility
learn.show_results(max_n=4, figsize=(35, 40))

# Create a SegmentationInterpretation object from the learner
interp = SegmentationInterpretation.from_learner(learn)

# Plot the top 4 losses (worst predictions), useful for error analysis,
# with large figure size for visibility
interp.plot_top_losses(k=4, figsize=(35, 40))


# Set the path where you want to save the model export
learn.path = Path('/work2/09601/mz868/frontera/modelSaves/')

# Filename for exported model
fname = 'spineSegModel6_combined_960.pkl'

# Export the trained model to a file using dill as the pickle module
learn.export(fname, pickle_module=dill)


In [ ]:
# Load a previously trained segmentation model using dill for serialization
learn = load_learner(
    '/work2/09601/mz868/frontera/modelSaves/spineSegModel4_820.pkl', 
    pickle_module=dill
)


In [ ]:

def pred2png(pred, location):
    """
    Save a predicted mask tensor as a PNG image at a specified location.
    
    Parameters:
    - pred: predicted mask tensor from the model
    - location: string containing the path info of the original DICOM/image file
    
    The output filename is generated by slicing the location string to extract part of the path.
    """
    
    # Previous attempts at output file naming (commented out):
    # out_file = "/work2/09601/mz868/frontera/960interpretations/" + location[-56:-4] + ".png"
    # out_file = "/work2/09601/mz868/frontera/860interpretations1/" + location[90:-4] + ".png"
    
    # Current output path uses a slice of 'location' starting at character 52 up to -4 (to strip extension),
    # then adds ".png" extension
    out_file = "/work2/09601/mz868/frontera/860dicominterpretations2/" + location[52:-4] + ".png"
    
    # Convert prediction tensor to numpy array
    j = pred.numpy()
    
    # This line sets all positive values in the mask to 255, and leaves zeros as zero
    # Note: np.where(j > 0, 255, j) replaces values >0 by 255, else keeps original j value
    # This effectively binarizes the mask to 0 or 255
    j_int = np.where(j > 0, 255, j)
    
    # Remove single-dimensional entries from the shape (e.g. convert (1, H, W) to (H, W))
    j = np.squeeze(j_int)
    
    # Save the processed mask as a PNG file
    cv2.imwrite(out_file, j)


In [ ]:
# Directory containing the JPG images to predict on
source_path = "/scratch1/09601/mz868/unsegmentedImages/dicom8202_2/"
# Collect all JPG files in the directory
image_files = glob.glob("/scratch1/09601/mz868/unsegmentedImages/dicom8202_2/*.jpg")

# Iterate over each image file path
for i in image_files:
    # Run the model's prediction on the image file path 'i'
    # Note: learn.predict expects a PIL image or path (fastai handles loading)
    predictionTest = learn.predict(i)
    
    # Extract predicted mask (first element of the returned tuple)
    prediction1 = predictionTest[0]
    
    # Save the prediction mask as a PNG using your custom function
    pred2png(prediction1, i)